# 03 — hikr.org: туристические треки → DEM slope → CatBoost speed-model

**Гипотеза:** туристические GPS-треки чище чем SAR — нет поисковых остановок.
DEM slope из COP30 вместо GPS altitude (±10-30м шум).
Результат сравниваем с notebook 02 (SAR данные).

In [ ]:
import re, pickle, time, warnings
import xml.etree.ElementTree as ET
from pathlib import Path
from math import radians, sin, cos, sqrt, atan2, degrees, atan, floor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio, rasterio.transform
from rasterio.merge import merge as _rmerge

warnings.filterwarnings("ignore")

try:
    from catboost import CatBoostRegressor
    HAS_CB = True
except ImportError:
    HAS_CB = False; print("pip install catboost")

try:
    from sklearn.model_selection import GroupKFold, GroupShuffleSplit
    from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
    HAS_SKL = True
except ImportError:
    HAS_SKL = False; print("pip install scikit-learn")

CACHE_DIR   = Path("cache")
MODELS_DIR  = Path("models")
DEM_TILE_DIR = Path("/Users/antonmikhaylyuk/Documents/magister_final_proj"
                    "/drone_code/data/terrain/tile_cache/dem")
for d in [CACHE_DIR, MODELS_DIR]:
    d.mkdir(exist_ok=True)

GPX_NS = "{http://www.topografix.com/GPX/1/1}"
print("OK")


## 1. Загрузка и EDA

In [ ]:
CSV_PATH = Path("gpx-tracks-from-hikr.org.csv")

# Метаданные без GPX-колонки для быстрого EDA
meta_cols = ["_id","length_2d","uphill","downhill","moving_time",
             "max_elevation","min_elevation","difficulty","start_time"]
meta = pd.read_csv(CSV_PATH, usecols=meta_cols)
print(f"Треков: {len(meta):,}")
print(f"moving_time notna: {meta['moving_time'].notna().sum():,}")
print()

# Скорость на уровне трека
meta["speed_kmh"] = meta["length_2d"] / meta["moving_time"] * 3.6
meta["elev_gain_m"] = meta["uphill"]
meta["length_km"]  = meta["length_2d"] / 1000

print(meta[["length_km","elev_gain_m","moving_time","speed_kmh"]].describe().round(2))
print()
print("difficulty:"); print(meta["difficulty"].value_counts().head(8))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
meta["speed_kmh"].clip(0, 10).hist(bins=50, ax=axes[0], color="#3b82f6")
axes[0].set_title("Скорость трека (км/ч)")
meta["length_km"].clip(0, 40).hist(bins=50, ax=axes[1], color="#22c55e")
axes[1].set_title("Длина (км)")
meta["elev_gain_m"].clip(0, 3000).hist(bins=50, ax=axes[2], color="#f97316")
axes[2].set_title("Набор высоты (м)")
plt.tight_layout(); plt.show()


## 2. Парсинг GPX → сегменты

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000
    dlat = radians(lat2-lat1); dlon = radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2*R*atan2(sqrt(a), sqrt(1-a))

def parse_gpx(gpx_str: str):
    """Возвращает list[dict] с lat, lon, ele_m, ts для каждой точки."""
    try:
        root = ET.fromstring(gpx_str)
    except ET.ParseError:
        return []
    pts = []
    for trkpt in root.iter(f"{GPX_NS}trkpt"):
        try:
            lat = float(trkpt.get("lat"))
            lon = float(trkpt.get("lon"))
            ele_tag = trkpt.find(f"{GPX_NS}ele")
            ele = float(ele_tag.text) if ele_tag is not None else None
            t_tag = trkpt.find(f"{GPX_NS}time")
            ts = pd.Timestamp(t_tag.text) if t_tag is not None else None
            pts.append({"lat": lat, "lon": lon, "ele": ele, "ts": ts})
        except Exception:
            continue
    return pts

SEG_DIST_MIN  = 1.0
SEG_DIST_MAX  = 1000.0
SEG_DT_MIN    = 1.0
SEG_DT_MAX    = 3600.0
SEG_SPEED_MIN = 0.3
SEG_SPEED_MAX = 15.0

DIFF_MAP = {
    "T1": 1, "T2": 2, "T3": 3, "T3+": 3, "T3-": 3,
    "T4": 4, "T4+": 4, "T4-": 4,
    "T5": 5, "T5+": 5, "T5-": 5,
    "T6": 6, "T6+": 6, "T6-": 6,
}

def diff_to_int(d):
    if pd.isna(d): return 3
    key = str(d).split(" - ")[0].strip()
    return DIFF_MAP.get(key, 3)

print("Парсинг GPX (это займёт 2-4 мин)...")
t0 = time.time()

seg_records = []
skipped = 0

for chunk in pd.read_csv(CSV_PATH, usecols=["_id","gpx","difficulty"], chunksize=500):
    for _, row in chunk.iterrows():
        tid   = row["_id"]
        diff  = diff_to_int(row["difficulty"])
        pts   = parse_gpx(row["gpx"])
        if len(pts) < 5:
            skipped += 1; continue
        for i in range(len(pts)-1):
            p1, p2 = pts[i], pts[i+1]
            if p1["ts"] is None or p2["ts"] is None: continue
            dt_s = (p2["ts"] - p1["ts"]).total_seconds()
            if not (SEG_DT_MIN <= dt_s <= SEG_DT_MAX): continue
            dist_m = haversine_m(p1["lat"],p1["lon"],p2["lat"],p2["lon"])
            if not (SEG_DIST_MIN <= dist_m <= SEG_DIST_MAX): continue
            spd = dist_m / dt_s * 3.6
            if not (SEG_SPEED_MIN <= spd <= SEG_SPEED_MAX): continue
            elev_diff = (round(p2["ele"]-p1["ele"],1)
                         if p1["ele"] is not None and p2["ele"] is not None else None)
            seg_records.append({
                "track_id": tid,
                "difficulty": diff,
                "lat1": p1["lat"], "lon1": p1["lon"],
                "lat2": p2["lat"], "lon2": p2["lon"],
                "lat_mid": (p1["lat"]+p2["lat"])/2,
                "lon_mid": (p1["lon"]+p2["lon"])/2,
                "dist_m": round(dist_m,2),
                "dt_s": dt_s,
                "elev_diff_m": elev_diff,
                "speed_kmh": spd,
            })

segs = pd.DataFrame(seg_records)
print(f"Готово за {(time.time()-t0)/60:.1f} мин")
print(f"Сегментов: {len(segs):,}  треков: {segs['track_id'].nunique():,}")
print(f"Пропущено треков: {skipped}")
print(segs[["dist_m","dt_s","speed_kmh","elev_diff_m"]].describe().round(2))


## 3. GPS slope (fallback) + DEM slope из COP30

In [ ]:
# GPS slope - пока как fallback
segs["slope_gps"] = segs.apply(
    lambda r: round(degrees(atan(abs(r["elev_diff_m"])/max(r["dist_m"],0.1))),3)
             if r["elev_diff_m"] is not None else None, axis=1
)
segs["going_up"] = (segs["elev_diff_m"].fillna(0) > 0).astype(int)
segs["slope_deg"] = segs["slope_gps"]  # заменим DEM ниже

# --- DEM slope из COP30 тайлов ---
def _tile_path(lat, lon):
    tl = floor(lat); to = floor(lon)
    ls = f"N{tl:02d}" if tl >= 0 else f"S{abs(tl):02d}"
    os_ = f"E{to:03d}" if to >= 0 else f"W{abs(to):03d}"
    return DEM_TILE_DIR / f"COP30_{ls}{os_}.tif"

def _dem_slope_batch(lats, lons):
    needed = set()
    for la, lo in zip(lats, lons):
        needed.add(_tile_path(la, lo))
    paths = [str(p) for p in needed if p.exists()]
    if not paths:
        return np.full(len(lats), np.nan)
    pad = 0.02
    bounds = (min(lons)-pad, min(lats)-pad, max(lons)+pad, max(lats)+pad)
    datasets = [rasterio.open(p) for p in paths]
    try:
        mosaic, tf = _rmerge(datasets, bounds=bounds)
    finally:
        for ds in datasets: ds.close()
    z = mosaic[0].astype(np.float32)
    nodata = rasterio.open(paths[0]).nodata
    if nodata is not None:
        z = np.where(z == nodata, np.nan, z)
    lat_c = (min(lats)+max(lats))/2
    from math import cos, radians, degrees, atan
    dx_m = abs(tf.a)*111320*cos(radians(lat_c))
    dy_m = abs(tf.e)*111320
    dz_dy, dz_dx = np.gradient(z, dy_m, dx_m)
    slope_arr = np.degrees(np.arctan(np.sqrt(dz_dx**2+dz_dy**2)))
    slope_arr = np.where(np.isnan(z), np.nan, slope_arr)
    rows, cols = rasterio.transform.rowcol(tf, list(lons), list(lats))
    rows = np.array(rows); cols = np.array(cols)
    h, w = slope_arr.shape
    valid = (rows>=0)&(rows<h)&(cols>=0)&(cols<w)
    result = np.full(len(lats), np.nan)
    result[valid] = slope_arr[rows[valid], cols[valid]]
    return result

DEM_CACHE_FILE = CACHE_DIR / "hikr_dem_slope.pkl"
if DEM_CACHE_FILE.exists():
    dem_slope = pickle.load(open(DEM_CACHE_FILE,"rb"))
    print("DEM slope из кэша")
else:
    dem_slope = np.full(len(segs), np.nan)
    segs_r = segs.reset_index(drop=True)
    tids = segs_r["track_id"].unique()
    for ti, tid in enumerate(tids):
        mask = (segs_r["track_id"]==tid).values
        idxs = np.where(mask)[0]
        mdf  = segs_r.iloc[idxs]
        try:
            vals = _dem_slope_batch(mdf["lat_mid"].values, mdf["lon_mid"].values)
            dem_slope[idxs] = vals
        except Exception as e:
            pass
        if (ti+1) % 1000 == 0:
            ok = (~np.isnan(dem_slope[:max(idxs)+1])).sum()
            print(f"  {ti+1}/{len(tids)} треков  DEM: {ok:,} сегм.")
    pickle.dump(dem_slope, open(DEM_CACHE_FILE,"wb"))

has_dem = ~np.isnan(dem_slope)
segs.loc[has_dem, "slope_deg"] = dem_slope[has_dem]
print(f"DEM slope: {has_dem.sum():,} / {len(segs):,} ({100*has_dem.mean():.1f}%)")
print(f"GPS slope медиана: {segs.loc[~has_dem,'slope_deg'].median():.2f}deg")
print(f"DEM slope медиана: {segs.loc[has_dem,'slope_deg'].median():.2f}deg")


## 4. Feature engineering + финальный датасет

In [ ]:
df = segs.dropna(subset=["slope_deg","speed_kmh"]).copy()

df["log_dist"]     = np.log1p(df["dist_m"])
df["slope_sq"]     = df["slope_deg"] ** 2
df["slope_x_up"]  = df["slope_deg"] * df["going_up"]

FEATURES = ["log_dist","slope_deg","slope_sq","elev_diff_m","going_up",
            "slope_x_up","difficulty"]
TARGET   = "speed_kmh"
CAT_FEATURES = ["going_up","difficulty"]

# Clip скорость 1-99 перцентили
v1, v99 = df[TARGET].quantile([0.01,0.99])
df = df[df[TARGET].between(v1,v99)]

for col in CAT_FEATURES:
    df[col] = df[col].astype(int)

print(f"Сегментов для модели: {len(df):,}")
print(f"Треков: {df['track_id'].nunique():,}")
print(f"\nTarget speed_kmh: медиана={df[TARGET].median():.2f}  std={df[TARGET].std():.2f}")
print(f"\nФичи: {FEATURES}")
print(df[FEATURES+[TARGET]].describe().round(3))

pickle.dump(df, open(CACHE_DIR/"hikr_model_df.pkl","wb"))
print("\nСохранено: cache/hikr_model_df.pkl")


## 5. CatBoost — GroupKFold по треку

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df["track_id"].values

for col in FEATURES:
    if X[col].isna().any():
        X[col] = X[col].fillna(X[col].median())

CB_PARAMS = dict(
    iterations=800, learning_rate=0.05, depth=6,
    loss_function="MAE", eval_metric="MAE",
    random_seed=42, verbose=0,
)

gkf = GroupKFold(n_splits=5)
cv_mae, cv_r2 = [], []

print("GroupKFold CV (5 folds по track_id):")
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    m = CatBoostRegressor(**CB_PARAMS, early_stopping_rounds=50)
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
          cat_features=CAT_FEATURES,
          eval_set=(X.iloc[val_idx], y.iloc[val_idx]),
          use_best_model=True)
    yp = m.predict(X.iloc[val_idx])
    fm = mean_absolute_error(y.iloc[val_idx], yp)
    fr = r2_score(y.iloc[val_idx], yp)
    cv_mae.append(fm); cv_r2.append(fr)
    print(f"  Fold {fold+1}: MAE={fm:.3f} км/ч  R^2={fr:.3f}")

print(f"\nCV:  MAE={np.mean(cv_mae):.3f} +/- {np.std(cv_mae):.3f}  "
      f"R^2={np.mean(cv_r2):.3f} +/- {np.std(cv_r2):.3f}")

print("\nФинальная модель (GroupShuffleSplit по track_id)...")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]
print(f"Train треков: {len(set(groups[tr_idx])):,}  Test треков: {len(set(groups[te_idx])):,}")
model = CatBoostRegressor(**CB_PARAMS, early_stopping_rounds=50)
model.fit(X_train, y_train, cat_features=CAT_FEATURES,
          eval_set=(X_test, y_test), use_best_model=True)
print("Готово.")
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

mae_tr = mean_absolute_error(y_train, y_train_pred)
r2_tr  = r2_score(y_train, y_train_pred)
mae_te = mean_absolute_error(y_test, y_test_pred)
r2_te  = r2_score(y_test, y_test_pred)

print(f"Train: MAE={mae_tr:.3f} км/ч  R^2={r2_tr:.3f}")
print(f"Test:  MAE={mae_te:.3f} км/ч  R^2={r2_te:.3f}")
print(f"Overfit gap  MAE: {mae_te-mae_tr:+.3f}  R^2: {r2_tr-r2_te:+.3f}")



## 6. Оценка + сравнение с notebook 02

In [ ]:
y_pred = model.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)**0.5
r2   = r2_score(y_test, y_pred)

from math import radians as _r
y_b1 = np.full(len(y_test), 3.5)
y_b2 = np.full(len(y_test), float(y_train.median()))
y_b3 = 6*np.exp(-3.5*np.abs(X_test["slope_deg"].apply(_r)+0.05))

print("=== РЕЗУЛЬТАТЫ ===")
print(f"CatBoost   MAE={mae:.3f} км/ч  RMSE={rmse:.3f}  R^2={r2:.3f}")
print(f"Baseline   3.5 km/h:     MAE={mean_absolute_error(y_test,y_b1):.3f}")
print(f"Baseline   train median: MAE={mean_absolute_error(y_test,y_b2):.3f}")
print(f"Baseline   Tobler:       MAE={mean_absolute_error(y_test,y_b3):.3f}")
print(f"Улучшение vs Tobler: {(mean_absolute_error(y_test,y_b3)-mae)/mean_absolute_error(y_test,y_b3)*100:.1f}%")
print()
print(f"CV R^2 (честный, по треку): {np.mean(cv_r2):.3f} +/- {np.std(cv_r2):.3f}")
print(f"Hold-out R^2:               {r2:.3f}")
print()
print("=== СРАВНЕНИЕ ===")
print("notebook 02 (SAR, GPS slope):    CV R^2~0.088  MAE~2.4 км/ч")
print(f"notebook 03 (hikr, DEM slope):   CV R^2~{np.mean(cv_r2):.3f}  MAE~{mae:.1f} км/ч")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fi = pd.DataFrame({"feature": FEATURES, "importance": model.get_feature_importance()}).sort_values("importance")
axes[0].barh(fi["feature"], fi["importance"], color="#3b82f6")
axes[0].set_title("Feature Importance")
for i,(_, row) in enumerate(fi.iterrows()):
    axes[0].text(row["importance"]+0.2, i, f"{row['importance']:.1f}", va="center", fontsize=8)

sidx = np.random.default_rng(0).choice(len(y_test), min(3000,len(y_test)), replace=False)
axes[1].scatter(np.array(y_test)[sidx], y_pred[sidx], s=2, alpha=0.3, c="#ef4444", rasterized=True)
lim = max(np.array(y_test).max(), y_pred.max())
axes[1].plot([0,lim],[0,lim],"k--",lw=1)
axes[1].set_xlabel("Факт (км/ч)"); axes[1].set_ylabel("Предсказание (км/ч)")
axes[1].set_title(f"Predicted vs Actual  (R^2={r2:.3f})")
plt.tight_layout(); plt.savefig("figures/hikr_model_eval.png", dpi=150, bbox_inches="tight"); plt.show()

model.save_model(str(MODELS_DIR/"travel_time_hikr.cbm"))
print("\nМодель: models/travel_time_hikr.cbm")
print("Применение: time_sec = dist_m / (model.predict([features]) / 3.6)")
